# Pose Model Evaluation

Evaluates the retrained YOLO11n-pose 5-kpt upper-body model on the held-out
**test** split (`data/pose/images/test`).

Reports:
- Box + Pose mAP (50, 50-95), precision, recall via Ultralytics `val`
- Per-keypoint **PCK** (Percentage of Correct Keypoints) at 0.05 / 0.1 / 0.2
  of shoulder-width
- Qualitative prediction overlays on sample test frames
- Inference latency / FPS

Picks the newest `best.pt` under `runs/pose/` automatically.

In [1]:
from __future__ import annotations

import glob
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO, settings

sns.set_theme(style="darkgrid", palette="Blues_d")

# No remote experiment logging during eval.
settings.update({k: False for k in (
    "clearml", "comet", "dvc", "hub", "mlflow",
    "neptune", "raytune", "tensorboard", "wandb",
)})

In [2]:
# ---- paths ----
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from fatigue_pipeline.constants import UPPER_BODY_KPT_NAMES  # noqa: E402

DATA_YAML = PROJECT_ROOT / "data/pose/dataset.yaml"
TEST_IMAGES = PROJECT_ROOT / "data/pose/images/test"
TEST_LABELS = PROJECT_ROOT / "data/pose/labels/test"

_cands = glob.glob(str(PROJECT_ROOT / "runs/pose/**/weights/best.pt"), recursive=True)
assert _cands, "No pose best.pt under runs/pose - train the model first."
WEIGHTS = max(_cands, key=lambda p: Path(p).stat().st_mtime)

KPT_NAMES = list(UPPER_BODY_KPT_NAMES)
SHOULDER_L, SHOULDER_R = 3, 4
DEVICE = 0  # 0 = first CUDA GPU; "cpu" if none

print("weights      :", WEIGHTS)
print("dataset yaml :", DATA_YAML)
print("test images  :", len(list(TEST_IMAGES.glob("*.jpg"))))
print("kpt names    :", KPT_NAMES)

weights      : c:\Users\jlord\OneDrive\Documents\Programming\FatigueSense\runs\pose\runs\pose\yolo11n_pose_upper55\weights\best.pt
dataset yaml : c:\Users\jlord\OneDrive\Documents\Programming\FatigueSense\data\pose\dataset.yaml
test images  : 424
kpt names    : ['nose', 'ear_left', 'ear_right', 'shoulder_left', 'shoulder_right']


## 1. Ultralytics val - mAP / P / R

Runs the built-in COCO-style pose evaluator on the test split.

In [3]:
model = YOLO(WEIGHTS)
metrics = model.val(data=str(DATA_YAML), split="test", device=DEVICE)

print(f"Box  mAP50    : {metrics.box.map50:.4f}")
print(f"Box  mAP50-95 : {metrics.box.map:.4f}")
print(f"Pose mAP50    : {metrics.pose.map50:.4f}")
print(f"Pose mAP50-95 : {metrics.pose.map:.4f}")
print(f"Box  P / R    : {metrics.box.mp:.4f} / {metrics.box.mr:.4f}")
print(f"Pose P / R    : {metrics.pose.mp:.4f} / {metrics.pose.mr:.4f}")

Ultralytics 8.4.25  Python-3.13.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11n-pose summary (fused): 110 layers, 2,654,632 parameters, 0 gradients, 6.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 26.66.4 MB/s, size: 168.3 KB)
val: Scanning C:\Users\jlord\OneDrive\Documents\Programming\FatigueSense\data\pose\labels\test... 424 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 424/424 326.6it/s 1.3s0.0s
val: New cache created: C:\Users\jlord\OneDrive\Documents\Programming\FatigueSense\data\pose\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 4.4it/s 6.1s0.1s
                   all        424        424          1          1      0.995      0.897          1          1      0.995      0.995
Speed: 1.0ms preprocess, 2.8ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to C:\Users\jlord\OneDrive\Docume

In [4]:
labels = ["Box mAP50", "Box mAP50-95", "Pose mAP50", "Pose mAP50-95"]
vals = [metrics.box.map50, metrics.box.map, metrics.pose.map50, metrics.pose.map]

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=labels, y=vals, ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("score")
ax.set_title("Pose model - test split mAP")
for i, v in enumerate(vals):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()

<Figure size 800x400 with 1 Axes>

## 2. Per-keypoint PCK

In [5]:
def _load_gt(label_path: Path, img_w: int, img_h: int):
    parts = label_path.read_text().split()
    if len(parts) < 6:
        return None
    kpts = np.array(parts[5:], dtype=float).reshape(-1, 3)
    xy = kpts[:, :2].copy()
    xy[:, 0] *= img_w
    xy[:, 1] *= img_h
    return xy, kpts[:, 2]


def _ref_dist(xy: np.ndarray, vis: np.ndarray) -> float | None:
    """Shoulder width if both shoulders labelled, else visible-kpt bbox diag."""
    if vis[SHOULDER_L] > 0 and vis[SHOULDER_R] > 0:
        d = float(np.linalg.norm(xy[SHOULDER_L] - xy[SHOULDER_R]))
        if d > 1.0:
            return d
    v = xy[vis > 0]
    if len(v) < 2:
        return None
    return float(np.linalg.norm(v.max(0) - v.min(0)))

In [6]:
THRESHOLDS = [0.05, 0.1, 0.2]
n_kpt = len(KPT_NAMES)
correct = {t: np.zeros(n_kpt) for t in THRESHOLDS}
total = np.zeros(n_kpt)

imgs = sorted(TEST_IMAGES.glob("*.jpg"))
skipped = 0

for img_path in imgs:
    lbl = TEST_LABELS / (img_path.stem + ".txt")
    if not lbl.exists():
        skipped += 1
        continue
    img = cv2.imread(str(img_path))
    if img is None:
        skipped += 1
        continue
    h, w = img.shape[:2]
    gt = _load_gt(lbl, w, h)
    if gt is None:
        skipped += 1
        continue
    gt_xy, gt_vis = gt
    ref = _ref_dist(gt_xy, gt_vis)
    if ref is None:
        skipped += 1
        continue

    res = model.predict(img, verbose=False, device=DEVICE)
    kp = res[0].keypoints
    if kp is None or kp.data is None or kp.data.shape[0] == 0:
        skipped += 1
        continue
    data = kp.data.cpu().numpy()  # (N, 5, 3)
    best = int(data[:, :, 2].mean(axis=1).argmax())
    pred_xy = data[best, :, :2]

    for k in range(n_kpt):
        if gt_vis[k] <= 0:
            continue
        total[k] += 1
        d = float(np.linalg.norm(pred_xy[k] - gt_xy[k])) / ref
        for t in THRESHOLDS:
            if d <= t:
                correct[t][k] += 1

pck = {
    t: np.divide(correct[t], total, out=np.zeros(n_kpt), where=total > 0)
    for t in THRESHOLDS
}

print(f"scored {int(total.max())} frames, skipped {skipped}")
for t in THRESHOLDS:
    print(f"PCK@{t:<4}: overall {pck[t].mean():.3f}")

scored 424 frames, skipped 0
PCK@0.05: overall 0.829
PCK@0.1 : overall 0.987
PCK@0.2 : overall 0.999


In [7]:
rows = []
for t in THRESHOLDS:
    for k, name in enumerate(KPT_NAMES):
        rows.append({"keypoint": name, "threshold": f"PCK@{t}", "value": pck[t][k]})
df_pck = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=df_pck, x="keypoint", y="value", hue="threshold", ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("PCK")
ax.set_title("Per-keypoint PCK (test split)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

df_pck.pivot(index="keypoint", columns="threshold", values="value").round(3)

<Figure size 1000x500 with 1 Axes>

threshold,PCK@0.05,PCK@0.1,PCK@0.2
keypoint,,,
ear_left,0.856,0.988,0.998
ear_right,0.934,0.993,1.000
nose,0.974,0.998,1.000
shoulder_left,0.682,0.967,0.998
shoulder_right,0.698,0.991,1.000


## 3. Qualitative overlays

In [8]:
import random

random.seed(0)
sample = random.sample(imgs, min(6, len(imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, img_path in zip(axes.ravel(), sample):
    res = model.predict(str(img_path), verbose=False, device=DEVICE)
    plotted = res[0].plot()  # BGR ndarray
    ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

<Figure size 1500x900 with 6 Axes>

## 4. Inference speed

Per-frame latency over the first 100 test frames (single image, batch=1).
Excludes the first warmup call.

In [9]:
_ = model.predict(str(sample[0]), verbose=False, device=DEVICE)  # warmup

bench = imgs[:100]
times = []
for img_path in bench:
    t0 = time.perf_counter()
    model.predict(str(img_path), verbose=False, device=DEVICE)
    times.append(time.perf_counter() - t0)

ms = np.array(times) * 1000.0
print(f"frames     : {len(bench)}")
print(f"latency    : {ms.mean():.1f} ms  +-{ms.std():.1f}")
print(f"throughput : {1000.0 / ms.mean():.1f} FPS")

frames     : 100
latency    : 20.7 ms  +-1.0
throughput : 48.4 FPS
